# Counter-ion pKa with "Machine learning meets pKa" (Baltruschat & Czodrowski, 2020)

Runs the published random-forest pKa model (Morgan fingerprints, trained on about 5,900 literature pKa values) on our counter-ion list.

**What it does**
1. Clones the authors' code from GitHub and applies two small fixes so it runs on current pandas and RDKit.
2. Trains the model (5-fold random forest, a few minutes).
3. **Checks it on ions where we already know the pKa**, so the error is reported before any prediction is used.
4. Predicts pKa for every ion that is not a permanent cation, and flags the ones the model cannot be trusted on.

**Known limits (from the authors' own pipeline)**
- Trained on pKa between 2 and 12 only, so strong acids (sulfonates, sulfates) come out wrong.
- Built for molecules with one ionizable site.
- Common small ions (acetate, chloride, sulfate and similar) are removed as "salts" and boron, nitro compounds are filtered out, so those get no prediction.

Upload `IL_ion_dataset_579_v5_pKa_fixed.csv` when asked.

In [1]:
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "rdkit", "scikit-learn", "pandas", "numpy"], check=True)

## 1. Get the authors' code and apply two small fixes

In [2]:
import os, subprocess, re
REPO = "Machine-learning-meets-pKa"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/czodrowskilab/Machine-learning-meets-pKa.git", REPO], check=True)

# Fix 1: modern pandas indexes a Series by label, not position -> force numpy arrays in the CV code
p = f"{REPO}/scripts/cv_regressor.py"
s = open(p).read()
if "np.asarray(x_data)" not in s:
    key = "kf = KFold(n_splits=self.n_folds, shuffle=self.shuffle, random_state=random_state)"
    assert key in s
    s = s.replace(key, "x_data = np.asarray(x_data)\n        y_data = np.asarray(y_data)\n        " + key, 1)
    open(p, "w").write(s)

# Fix 2: new RDKit removed MolStandardize.standardize (old MolVS classes) -> use a small compatibility shim
p = f"{REPO}/scripts/gen_clean_mono_dataset.py"
s = open(p).read()
old = "from rdkit.Chem.MolStandardize.standardize import TautomerCanonicalizer, Uncharger"
if old in s:
    new = """from rdkit.Chem.MolStandardize import rdMolStandardize as _rms
Uncharger = _rms.Uncharger
class TautomerCanonicalizer:
    def __init__(self, max_tautomers=1000):
        self._te = _rms.TautomerEnumerator(); self._te.SetMaxTautomers(max_tautomers)
    def canonicalize(self, mol):
        return self._te.Canonicalize(mol)"""
    open(p, "w").write(s.replace(old, new))
print("code ready")

code ready


## 2. Train the model (skipped if a trained model is already there)

In [3]:
os.chdir(REPO)
if not os.path.exists("RF_CV_FMorgan3_pKa.pkl"):
    nproc = os.cpu_count() or 2
    subprocess.run([sys.executable, "train_model.py", "--no-openeye", "--num-processes", str(nproc)], check=True)
print("model file:", os.path.getsize("RF_CV_FMorgan3_pKa.pkl") // 1_000_000, "MB")

model file: 19 MB


## 3. Load our ion dataset

In [4]:
import pandas as pd, numpy as np
from rdkit import Chem
from rdkit.Chem import PandasTools

NAME = "IL_ion_dataset_579_v5_pKa_fixed.csv"
if "google.colab" in sys.modules:
    from google.colab import files
    up = files.upload()
    NAME = list(up.keys())[0]
    ions = pd.read_csv(NAME)
else:
    ions = pd.read_csv(os.environ.get("ION_CSV", NAME))
print(len(ions), "ions")
print(ions.pKa_status.value_counts())

579 ions
pKa_status
permanent cation (no pKa)    220
our DFT                      116
measured                     115
web lookup                    71
our DFT (new run)             36
no pKa yet                    21
Name: count, dtype: int64


## 4. Helper: write ions to an SDF, run the authors' predictor, read the result back

In [5]:
def run_model(df, tag):
    """df needs columns ion_id and smiles. Returns ion_id -> predicted pKa (ions the pipeline drops are simply missing)."""
    w = Chem.SDWriter(f"{tag}_in.sdf")
    for r in df.itertuples():
        m = Chem.MolFromSmiles(r.smiles)
        if m is None: continue
        m.SetProp("_Name", r.ion_id); w.write(m)
    w.close()
    subprocess.run([sys.executable, "predict_sdf.py", f"{tag}_in.sdf", f"{tag}_out.sdf", "--no-openeye"],
                   check=True, capture_output=True)
    o = PandasTools.LoadSDF(f"{tag}_out.sdf")
    return dict(zip(o["ID"], o["pKa_prediction"].astype(float)))

## 5. Check first: measured ions (we know their pKa)

Measured = the `measured` ions plus web lookups that are marked as measured. If the error here is large, the predictions below deserve the same doubt.

In [6]:
conf = ions.pKa_confidence.fillna("").str.lower()
known = ions[(ions.pKa_status == "measured") | ((ions.pKa_status == "web lookup") & conf.str.startswith("measured |"))].copy()
pred = run_model(known, "val")
known["pred"] = known.ion_id.map(pred)
known["abs_err"] = (known.pKa - known.pred).abs()
ok = known[known.pred.notna()]
inr = ok[(ok.pKa >= 2) & (ok.pKa <= 12)]
outr = ok[(ok.pKa < 2) | (ok.pKa > 12)]
print(f"ions with a measured value: {len(known)}   model gave a value for: {len(ok)}")
print(f"all:                 MAE {ok.abs_err.mean():.2f}   median {ok.abs_err.median():.2f}")
print(f"true pKa 2 to 12:    n={len(inr)}  MAE {inr.abs_err.mean():.2f}   median {inr.abs_err.median():.2f}")
print(f"true pKa outside:    n={len(outr)}  MAE {outr.abs_err.mean():.2f}")
print(ok.groupby("ion_type").abs_err.agg(["count", "mean"]).round(2))
print("\nLargest errors:")
print(ok.sort_values("abs_err", ascending=False).head(10)[["ion_id", "name", "pKa", "pred", "abs_err"]].round(2).to_string(index=False))

ions with a measured value: 153   model gave a value for: 138
all:                 MAE 1.41   median 0.60
true pKa 2 to 12:    n=129  MAE 1.10   median 0.51
true pKa outside:    n=9  MAE 5.74
          count  mean
ion_type             
anion        94  1.53
cation       44  1.14

Largest errors:
  ion_id                               name   pKa  pred  abs_err
  AN_135                  dimethylphosphate  1.25  8.51     7.26
  AN_015 1,1-dioxo-1,2-benzothiazol-3-olate  1.60  8.36     6.76
 AN_N003                       saccharinate  1.60  8.36     6.76
  AN_041                   benzenesulfonate -2.80  3.50     6.30
  AN_109                           docusate -2.00  3.74     5.74
  AN_004                          glycinate  9.60  4.05     5.55
 CAT_023             Tetramethylguanidinium 13.60  8.15     5.45
CAT_N045   argininium (protonated arginine) 12.48  7.05     5.43
 CAT_028                       methioninium  2.15  7.02     4.87
CAT_N003                      tryptophanium  2.42  7.

## 5b. Metrics: MAE, RMSE and R² on the authors' two external test sets and on our measured ions

The two test sets ship with the repository and were removed from the training data by the authors. The paper reports, for this random forest (FCFP6, 4096 bit): 5-fold CV MAE 0.708 / RMSE 1.094 / R² 0.797; Novartis MAE 1.147 / RMSE 1.513 / R² 0.569; AvLiLuMoVe MAE 0.532 / RMSE 0.785 / R² 0.889 (prepared with OpenEye QUACPAC; here RDKit is used, so numbers can differ).

In [7]:
def scores(t, p):
    t, p = np.asarray(t, float), np.asarray(p, float)
    e = p - t
    return dict(MAE=np.abs(e).mean(), RMSE=np.sqrt((e**2).mean()), R2=1 - (e**2).sum() / ((t - t.mean())**2).sum())

def run_model_sdf(path, tag):
    e = PandasTools.LoadSDF(path)
    w = Chem.SDWriter(f"{tag}_in.sdf")
    for mol, i in zip(e.ROMol, e.ID):
        if mol is None: continue
        mol.SetProp("_Name", str(i)); w.write(mol)
    w.close()
    subprocess.run([sys.executable, "predict_sdf.py", f"{tag}_in.sdf", f"{tag}_out.sdf", "--no-openeye"], check=True, capture_output=True)
    o = PandasTools.LoadSDF(f"{tag}_out.sdf")
    m = e[["ID", "pKa"]].merge(o[["ID", "pKa_prediction"]], on="ID")
    return len(e), m.pKa.astype(float), m.pKa_prediction.astype(float)

METRICS = []
for name, f in [("Novartis", "datasets/novartis_cleaned_mono_unique_notraindata.sdf"),
                ("AvLiLuMoVe", "datasets/AvLiLuMoVe_cleaned_mono_unique_notraindata.sdf")]:
    n_all, t, p = run_model_sdf(f, name)
    METRICS.append(dict(model="RF (ML meets pKa)", test_set=name, n=len(t), **scores(t, p)))
    print(f"{name}: {len(t)} of {n_all} predicted")
for label, df in [("our measured ions (all)", ok), ("our measured ions (true pKa 2-12)", inr), ("our measured ions (true pKa outside 2-12)", outr)]:
    METRICS.append(dict(model="RF (ML meets pKa)", test_set=label, n=len(df), **scores(df.pKa, df.pred)))
metrics_df = pd.DataFrame(METRICS).round(3)
metrics_df

Novartis: 280 of 280 predicted


AvLiLuMoVe: 123 of 123 predicted


,model,test_set,n,MAE,RMSE,R2
0,RF (ML meets pKa),Novartis,280,1.444,1.801,0.389
1,RF (ML meets pKa),AvLiLuMoVe,123,0.964,1.430,0.633
2,RF (ML meets pKa),our measured ions (all),138,1.407,2.218,0.501
3,RF (ML meets pKa),our measured ions (true pKa 2-12),129,1.105,1.697,0.632
4,RF (ML meets pKa),our measured ions (true pKa outside 2-12),9,5.736,5.844,-0.121


## 5c. Reproduction check: why the external numbers above are weaker than the paper

The two test sets in the repository were prepared with OpenEye QUACPAC (licensed). The model above is trained on the free RDKit-prepared set, so the test molecules are prepared differently from its training molecules. To show the code itself reproduces the paper, a smaller random forest (100 trees, one fit) is trained on the **OpenEye-prepared** training set and scored on the test sets as shipped. Takes about 2 minutes.

In [8]:
from sklearn.ensemble import RandomForestRegressor
from rdkit.Chem import AllChem
fpv = lambda mol: np.array(AllChem.GetMorganFingerprintAsBitVect(mol, radius=3, nBits=4096, useFeatures=True))
tr_oe = PandasTools.LoadSDF("datasets/combined_training_datasets_unique.sdf"); tr_oe = tr_oe[tr_oe.ROMol.notna()]
rf_oe = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=24).fit(
    np.array([fpv(x) for x in tr_oe.ROMol]), tr_oe.pKa.astype(float).values)
for name, f in [("Novartis", "datasets/novartis_cleaned_mono_unique_notraindata.sdf"),
                ("AvLiLuMoVe", "datasets/AvLiLuMoVe_cleaned_mono_unique_notraindata.sdf")]:
    e = PandasTools.LoadSDF(f)
    s_ = scores(e.pKa.astype(float), rf_oe.predict(np.array([fpv(x) for x in e.ROMol])))
    METRICS.append(dict(model="RF reproduction check (OpenEye-prepared training, 100 trees)", test_set=name, n=len(e), **s_))
metrics_df = pd.DataFrame(METRICS).round(3)
metrics_df

[14:25:47] Warning: ambiguous stereochemistry - overlapping neighbors  - at atom 13 ignored


[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerat

[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerat

[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerat

[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerat

[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerator
[14:25:48] DEPRECATION WARNING: please use MorganGenerat

... (52 outputs left out)

[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerat

[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerator
[14:27:59] DEPRECATION WARNING: please use MorganGenerat

[14:28:00] DEPRECATION WARNING: please use MorganGenerator
[14:28:00] DEPRECATION WARNING: please use MorganGenerator
[14:28:00] DEPRECATION WARNING: please use MorganGenerator
[14:28:00] DEPRECATION WARNING: please use MorganGenerator
[14:28:00] DEPRECATION WARNING: please use MorganGenerator
[14:28:00] DEPRECATION WARNING: please use MorganGenerator
[14:28:00] DEPRECATION WARNING: please use MorganGenerator
[14:28:00] DEPRECATION WARNING: please use MorganGenerator
[14:28:00] DEPRECATION WARNING: please use MorganGenerator
[14:28:00] DEPRECATION WARNING: please use MorganGenerator
[14:28:00] DEPRECATION WARNING: please use MorganGenerator
[14:28:00] DEPRECATION WARNING: please use MorganGenerator
[14:28:00] DEPRECATION WARNING: please use MorganGenerator
[14:28:00] DEPRECATION WARNING: please use MorganGenerator
[14:28:00] DEPRECATION WARNING: please use MorganGenerator
[14:28:00] DEPRECATION WARNING: please use MorganGenerator
[14:28:00] DEPRECATION WARNING: please use MorganGenerat

,model,test_set,n,MAE,RMSE,R2
0,RF (ML meets pKa),Novartis,280,1.444,1.801,0.389
1,RF (ML meets pKa),AvLiLuMoVe,123,0.964,1.430,0.633
2,RF (ML meets pKa),our measured ions (all),138,1.407,2.218,0.501
3,RF (ML meets pKa),our measured ions (true pKa 2-12),129,1.105,1.697,0.632
4,RF (ML meets pKa),our measured ions (true pKa outside 2-12),9,5.736,5.844,-0.121
5,RF reproduction check (OpenEye-prepared traini...,Novartis,280,1.141,1.517,0.566
6,RF reproduction check (OpenEye-prepared traini...,AvLiLuMoVe,123,0.511,0.772,0.893


## 6. Predict every ion that is not a permanent cation

In [9]:
todo = ions[ions.pKa_status != "permanent cation (no pKa)"].copy()
todo["ml_pKa"] = todo.ion_id.map(run_model(todo, "all"))

strong = [Chem.MolFromSmarts(x) for x in ["[SX4](=O)(=O)[O-,OH]", "[SX4](=O)(=O)[N-,NH]C=O", "P(=O)([O-,OH])O"]]
def flag(r):
    m = Chem.MolFromSmiles(r.smiles)
    if m is None: return "no structure"
    if pd.isna(r.ml_pKa):
        if m.HasSubstructMatch(Chem.MolFromSmarts("[B,Si,Se]")): return "not predicted: boron/silicon/selenium is filtered out"
        if m.HasSubstructMatch(Chem.MolFromSmarts("[N+](=O)[O-]")): return "not predicted: nitro group is filtered out"
        return "not predicted: removed as a common salt ion or filtered"
    if any(m.HasSubstructMatch(q) for q in strong): return "do not trust: strong-acid group (model trained on pKa 2 to 12 only)"
    if m.GetNumHeavyAtoms() > 40: return "low confidence: large molecule, may have several ionizable sites"
    return "ok (within the model's range)"
todo["ml_flag"] = todo.apply(flag, axis=1)
print(todo.ml_flag.value_counts())

# how close is it to our DFT values?
d = todo[todo.pKa_status.isin(["our DFT", "our DFT (new run)"]) & todo.ml_pKa.notna()]
print(f"\nvs our DFT values: n={len(d)}  MAE {(d.pKa - d.ml_pKa).abs().mean():.2f}  median {(d.pKa - d.ml_pKa).abs().median():.2f}")

ml_flag
ok (within the model's range)                                          277
do not trust: strong-acid group (model trained on pKa 2 to 12 only)     50
not predicted: removed as a common salt ion or filtered                 17
not predicted: boron/silicon/selenium is filtered out                   10
not predicted: nitro group is filtered out                               5
Name: count, dtype: int64

vs our DFT values: n=150  MAE 2.58  median 1.09


## 7. The ions that still had no pKa

In [10]:
targets = ["AN_N001", "AN_N002", "CAT_N021", "CAT_N036", "AN_N028", "AN_N045"]
t = todo[todo.ion_id.isin(targets)][["ion_id", "name", "ion_type", "ml_pKa", "ml_flag"]].round(2)
t

,ion_id,name,ion_type,ml_pKa,ml_flag
416,AN_N001,acesulfamate,anion,7.70,do not trust: strong-acid group (model trained...
418,AN_N002,geranate,anion,4.61,ok (within the model's range)
446,CAT_N021,didecylmethylammonium,cation,9.70,ok (within the model's range)
463,CAT_N036,clofaziminium (protonated clofazimine),cation,4.90,ok (within the model's range)
501,AN_N028,5-nitrobarbiturate,anion,NaN,not predicted: nitro group is filtered out
547,AN_N045,tetraphenylborate,anion,NaN,not predicted: boron/silicon/selenium is filte...


## 8. Save

In [11]:
out = ions.merge(todo[["ion_id", "ml_pKa", "ml_flag"]], on="ion_id", how="left")
out["ml_validation_MAE_2to12"] = round(inr.abs_err.mean(), 2)
OUT = "IL_ion_pKa_ML_meets_pKa.csv"
out.to_csv(os.path.join("..", OUT) if not "google.colab" in sys.modules else OUT, index=False)
metrics_df.to_csv(os.path.join("..", "RF_ML_meets_pKa_metrics.csv") if not "google.colab" in sys.modules else "RF_ML_meets_pKa_metrics.csv", index=False)
if "google.colab" in sys.modules:
    files.download(OUT); files.download("RF_ML_meets_pKa_metrics.csv")
print("saved", OUT, len(out), "rows")

saved IL_ion_pKa_ML_meets_pKa.csv 579 rows
